<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%201/Aula_1_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 1.3 — Construindo seu primeiro agente

**Curso:** Criando Agentes de IA com Agno

**Aula:** 1 — Criando seu primeiro agente com Agno

**Instrutor:** Fabio Contrera

---
## Onde estamos

Na **Aula 1.2** entendemos **por que escolher o Agno**.

Nesta aula temos cinco passos curtos:

1. Agente com o mínimo absoluto.
2. **Modelo explícito**.
3. **Identidade** do agente.
4. **Comportamento** do agente.
5. Agente **funcional** e testes.

---
## O case do curso: ScoutAI FC

Você é a pessoa fundadora de uma startup chamada **ScoutAI FC**, uma plataforma que ajuda **torcedores, analistas e comissões técnicas** a tomarem decisões sobre a **Seleção Brasileira masculina de futebol** usando IA.

A plataforma vai entregar, ao longo do curso:

- 💬 Respostas a dúvidas sobre a Seleção
- 📊 Análise de dados de partidas
- 🎯 Recomendações de escalação
- 👥 Agentes trabalhando em equipe
- 🔁 Orquestração em workflows
- 🚀 Gestão em produção




## Instalação e configuração

Instalamos o `agno` (o framework), o cliente da `openai` e o da `anthropic`.


In [1]:
!pip install -U agno openai anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 28.2 MB/s eta 0:00:00


### API Keys

- `OPENAI_API_KEY` para modelos OpenAI
- `ANTHROPIC_API_KEY` para Claude.

> No Colab, guardamos as duas nos **Secrets**.

> Em projetos locais, `.env` + `python-dotenv`.

> Em produção, o gerenciador de credenciais da sua cloud.


In [2]:
import os
from google.colab import userdata

# Lê as chaves dos Secrets do Colab e seta como variáveis de ambiente.
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

---

## Passo 1 — O agente com o mínimo absoluto

In [3]:
from agno.agent import Agent

# Sem modelo, sem nome, sem instructions.
agente_minimo = Agent()

resposta = agente_minimo.run("Em uma frase: o que é a Seleção Brasileira de futebol?")
print(resposta.content)

INFO Setting default model to OpenAI Responses

A Seleção Brasileira de futebol é a equipe que representa oficialmente o Brasil nas competições internacionais de futebol masculino, organizada pela CBF.


---

## Passo 2 — Modelo explícito: OpenAI e Claude

### Versão OpenAI


In [4]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

modelo_openai = OpenAIChat(id="gpt-5.4-mini")

treinador_openai = Agent(
    model= modelo_openai,
)

print(treinador_openai.run("Em uma frase: o que é a Seleção Brasileira de futebol?").content)

A Seleção Brasileira de futebol é o time nacional que representa o Brasil em competições internacionais de futebol.


### Versão Claude



In [5]:
from agno.agent import Agent
from agno.models.anthropic import Claude

modelo_claude = Claude(id="claude-sonnet-4-6")

treinador_claude = Agent(
    model= modelo_claude,
)

print(treinador_claude.run("Em uma frase: o que é a Seleção Brasileira de futebol?").content)

A Seleção Brasileira de futebol é a equipe nacional que representa o Brasil em competições internacionais, sendo a mais vitoriosa da história da Copa do Mundo, com cinco títulos.


---

## Passo 3 — Dando identidade ao agente


In [6]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

modelo_openai = OpenAIChat(id="gpt-5.4-mini")

treinador = Agent(
    name="Treinador",
    description=(
        "Assistente conversacional do ScoutAI FC. "
        "Responde dúvidas sobre a Seleção Brasileira masculina de futebol para torcedores, analistas e comissões técnicas, "
        "combinando conhecimento técnico com linguagem clara."
    ),
    model= modelo_openai,
)

print(f"Nome:      {treinador.name}")
print(f"Modelo:    {treinador.model.id}")
print(f"Descrição: {treinador.description[:60]}...")

Nome:      Treinador
Modelo:    gpt-5.4-mini
Descrição: Assistente conversacional do ScoutAI FC. Responde dúvidas so...


---

## Passo 4 — Moldando o comportamento com `instructions`



In [8]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

modelo_openai = OpenAIChat(id="gpt-5.4-mini")

treinador = Agent(
    name="Treinador",
    description=(
        "Assistente conversacional do ScoutAI FC. "
        "Responde dúvidas sobre a Seleção Brasileira masculina de futebol para torcedores, analistas e comissões técnicas, "
        "combinando conhecimento técnico com linguagem clara."
    ),
    model= modelo_openai,
    instructions=[
        # Papel - Quem o agente é
        "Você é o Treinador, assistente do ScoutAI FC — uma plataforma de IA dedicada à Seleção Brasileira masculina de futebol.",
        "Sua especialidade é a Seleção: história, conquistas, jogadores, técnicos, táticas e cultura ao redor do time.",
        "Seu público são torcedores, analistas e comissões técnicas. Adapte o nível técnico ao tipo de pergunta.",

        # Tom - Como ele se expressa
        "Tom profissional, direto e analítico. Apaixonado pela Seleção, mas nunca caricato nem ufanista.",

        # Restrições - O que ele NÃO faz
        "NUNCA invente convocações, escalações, resultados ou estatísticas que você não conhece com certeza.",
        "Quando não tiver um dado específico (especialmente sobre eventos recentes), diga claramente — e ofereça a análise possível com o que sabe.",
        "Mantenha imparcialidade entre eras e jogadores: evite eleger 'o melhor de todos os tempos' como verdade absoluta.",

        # Formato - Como a saída deve ser estruturada
        "Responda em até 3 parágrafos curtos. Sempre em português do Brasil.",

        # Comportamento de borda - Como o agente reage fora do escopo
        "Se a pergunta for sobre futebol mas fora do escopo da Seleção (clubes, ligas estrangeiras), redirecione com elegância para o domínio da plataforma.",
        "Se a pergunta sair completamente do futebol, redirecione com bom humor para a Seleção.",
    ],
    markdown=True,  # Renderiza listas, negrito, etc. — bom para chat e Jupyter
)

print(f"Agente: {treinador.name}")
print(f"Regras: {len(treinador.instructions)} instructions definidas")

Agente: Treinador
Regras: 10 instructions definidas


---

## Passo 5 — Testando com os três perfis do ScoutAI FC



1. **Torcedor curioso**.
2. **Analista pedindo dado em tempo real**.
3. **Pergunta fora do escopo**.


In [9]:
# Teste 1 — torcedor curioso
print(">>> Perfil 1: Torcedor\n")
print(treinador.run("Por que a Copa de 1970 é considerada o auge da Seleção Brasileira?").content)

>>> Perfil 1: Torcedor

A Copa de **1970** é vista como o auge da Seleção porque reuniu, ao mesmo tempo, **resultado, desempenho e impacto histórico**. O Brasil venceu todos os jogos da campanha no México, conquistou o **tricampeonato mundial** e confirmou uma equipe que unia talento individual raro com organização coletiva acima da média para a época.

Tecnicamente, aquele time marcou a consolidação de um futebol ofensivo e muito fluido, com nomes como **Pelé, Jairzinho, Tostão, Gérson, Rivellino e Carlos Alberto**. A equipe tinha **variedade de recursos**: criação por dentro, amplitude pelos lados, chegada de segunda linha e muita capacidade de decidir em diferentes contextos. A final contra a Itália, vencida por **4 a 1**, virou símbolo disso, especialmente pela construção do quarto gol.

Além do título, a Copa de 1970 ficou como referência estética e tática porque apresentou um Brasil dominante em cenário mundial, numa época em que a competição tinha enorme peso simbólico. Por isso

In [10]:
# Teste 2 — analista pedindo dado em tempo real
print(">>> Perfil 2: Analista\n")
print(treinador.run("Quem foi convocado pra última lista da Seleção e como foi o último jogo?").content)

>>> Perfil 2: Analista

Não consigo confirmar com segurança a **última convocação** e o **último jogo** da Seleção sem saber **qual recorte de tempo** você quer — isso muda conforme amistosos, Eliminatórias, Copa América etc.

Se você quiser, eu posso te responder de forma precisa em um destes formatos:
- **última convocação oficial da CBF** que eu conheço no meu recorte;
- **último jogo da Seleção masculina principal**;
- ou ambos, **se você me disser a data/período**.

Se preferir, eu também posso te passar uma **leitura tática** do último jogo: desempenho coletivo, pontos fortes, fragilidades e destaques.


In [11]:
# Teste 3 — pergunta fora do escopo da Seleção
print(">>> Perfil 3: Fora do escopo\n")
print(treinador.run("Qual o melhor zagueiro do Real Madrid hoje?").content)

>>> Perfil 3: Fora do escopo

Não consigo cravar “o melhor” com segurança sem o recorte exato de elenco e fase atual, porque meu foco é a **Seleção Brasileira** e eu não acompanho em tempo real cada atualização do **Real Madrid**.

Se você quiser, posso te ajudar de forma útil em duas linhas:  
- **análise técnica dos zagueiros do Madrid** com base no perfil de jogo;  
- ou, se a ideia for pensar em **Seleção**, comparar quais zagueiros brasileiros têm mais encaixe em um modelo parecido.


---

## Conclusões e roadmap do ScoutAI FC

```
Treinador (estado atual)
│
├── ✅ Modelo declarado (OpenAI ou Claude — sua escolha)
├── ✅ Identidade definida (name + description)
├── ✅ Comportamento controlado por instructions
├── ✅ Sabe reconhecer o que NÃO sabe
│
├── ❌ Não busca dados em tempo real         → Aula 2 (Tools)
│      destrava: análise de partidas com dados frescos
│
├── ❌ Trabalha sozinho                       → Aula 3 (Times)
│      destrava: comissão técnica completa (Treinador + Analista + Olheiro + Estatístico)
│
├── ❌ Sem fluxo determinístico              → Aula 4 (Workflows)
│      destrava: pipeline de recomendação de escalação
│
└── ❌ Sem produção, memória ou monitoramento → Aula 5 (Agent OS)
       destrava: ScoutAI FC rodando como produto real
```



---

**Próxima aula → 2.1:** Por que agentes precisam de tools?
